# Indian Road Accidents 2022–2025 — Complete EDA

This notebook re-runs the project's analysis on the **full** dataset using the same tested modules that power the dashboard (`src/`). Nothing here is hard-coded: every number is computed from `data/processed/cleaned_road_accidents.csv`.

**Contents** 1 Profile & provenance · 2 Cleaning · A Overall · B Geography · C Temporal · D Severity · E Causes · F Weather · G Infrastructure · H Festivals · I Risk score · J Statistics · Insights

> **Terminology.** *accidents* = records · *fatal accidents* = severity 'fatal' · *fatality rate* = fatal ÷ accidents in the group · *casualties* = persons killed or injured (deaths are not separated, so fatalities cannot be counted).

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)

from src import analysis as A, config, visualizations as V
from src.data_loader import load_raw, ensure_clean_data
from src.profiling import profile_dataset, RISK_FORMULA_TEXT
from src.insights import generate_insights

raw = load_raw()
df = ensure_clean_data()
print(f"raw {raw.shape}  ·  cleaned {df.shape}")

raw (20000, 24)  ·  cleaned (20000, 54)


## 1. Dataset profile and provenance
The profile checks schema, types, missing values, duplicates, date/time formats, coordinates, logical consistency, derived columns and statistical fingerprints of simulation.

In [2]:
p = profile_dataset(raw)
pd.DataFrame(p["columns"])[["column", "dtype", "role", "missing", "missing_pct", "unique", "examples"]]

,column,dtype,role,missing,missing_pct,unique,examples
0,accident_id,int64,identifier,0,0.000,20000,"[0, 1, 2, 3, 4]"
1,city,str,categorical,0,0.000,8,"[Pune, Mumbai, Chandigarh, Chennai, Delhi]"
2,state,str,categorical,0,0.000,7,"[Maharashtra, Punjab, Tamil Nadu, Delhi, Karna..."
3,latitude,float64,numeric,0,0.000,19907,"[18.680827, 18.817732, 19.096889, 30.787805, 1..."
4,longitude,float64,numeric,0,0.000,19927,"[73.930388, 72.790846, 72.819424, 76.847507, 8..."
5,date,str,date/time,0,0.000,1201,"[2023-10-22, 2023-05-21, 2024-07-10, 2025-03-3..."
6,time,str,date/time,0,0.000,24,"[5:00, 4:00, 13:00, 11:00, 16:00]"
7,hour,int64,numeric,0,0.000,24,"[5, 4, 13, 11, 16]"
8,day_of_week,str,categorical,0,0.000,7,"[Sunday, Wednesday, Thursday, Monday, Friday]"
9,is_weekend,int64,binary,0,0.000,2,"[1, 0]"


In [3]:
print(p["provenance"]["verdict"], "\n")
for s in p["provenance"]["signals"]:
    print("•", s)

Likely SIMULATED / SYNTHETIC. Treat all findings as properties of this dataset, not of Indian road safety. 

• 8 of 9 categorical/temporal marginals are statistically indistinguishable from a uniform distribution (chi-square, α=0.05).
• Every timestamp is exactly on the hour (minutes = 00).
• risk_score takes only 18 discrete values (step 0.05).
• risk_score is reproduced exactly (100.0% of rows) by a simple additive rule: risk_score = min(1, 0.10 + 0.20·[severity = fatal] + 0.20·[traffic_density = high] + 0.15·[is_peak_hour] + 0.15·[weather ∈ {fog, rain}] + 0.20·[visibility = low] + 0.10·[festival recorded]).
• For every fatal accident, casualties == vehicles_involved exactly; casualties never exceed vehicles in any record — a generation rule, not an observed pattern.
• 1,725 'major' accidents record zero casualties (logically inconsistent with a major-injury classification).
• Festival labels cover only some years; real festival days with records but no label: Diwali 2022-10-24, Holi

In [4]:
pd.DataFrame(p["uniformity"]).T

,categories,chi2,p_value,max_to_min_ratio,consistent_with_uniform
city,8,12.8008,0.077113,1.069738,True
hour,24,22.8256,0.470993,1.129771,True
day_of_week,7,10.6074,0.101295,1.088041,True
cause,5,0.6175,0.961099,1.013085,True
weather,3,0.2677,0.874721,1.008593,True
road_type,3,1.4203,0.49157,1.019498,True
lanes,6,4.1986,0.521191,1.03519,True
traffic_density,3,132.6769,0.0,1.198,False
month (complete months),39,50.035523,0.091482,1.261745,True


In [5]:
pd.DataFrame(p["derived_columns"])

,column,derived_from,agreement_pct
0,hour,time,100.000
1,day_of_week,date,100.000
2,is_weekend,date,100.000
3,is_peak_hour,"hour ∈ [8, 9, 10, 17, 18, 19]",100.000
4,visibility,weather (modal mapping),83.385
5,risk_score,composite; strongest single input = visibility...,NaN


In [6]:
print(p["risk_score"]["formula_check"])
pd.crosstab(raw["weather"], raw["visibility"])

{'formula': 'risk_score = min(1, 0.10 + 0.20·[severity = fatal] + 0.20·[traffic_density = high] + 0.15·[is_peak_hour] + 0.15·[weather ∈ {fog, rain}] + 0.20·[visibility = low] + 0.10·[festival recorded])', 'exact_match_pct': 100.0, 'max_abs_difference': 1.1102230246251565e-16}


visibility,high,low,medium
weather,,,
clear,6690,0,0
fog,0,6633,0
rain,0,3354,3323


## 2. Cleaning log
Every action of `src/preprocessing.py`. Only exact duplicates may be removed; impossible values become missing (never deleted); outliers are flagged, not dropped.

In [7]:
pd.read_json(config.CLEANING_LOG_JSON)

,step,column,action,rows_affected,rationale
0,load,<all>,loaded raw records,20000,Full dataset — no sampling.
1,duplicates,accident_id,verified unique,0,No duplicate ids found.
2,dates,time,normalised 'H:MM' → 'HH:00',20000,Raw mixes '5:00' and '13:00'; all times are on...
3,coordinates,latitude/longitude,validated against India bounding box and city ...,0,Invalid coordinates are excluded from maps via...
4,missing,festival,missing → 'No Festival' (in festival_name); ra...,19885,Missing here is structural (no festival that d...
5,features,condition_risk,risk_score recomputed without its severity term,20000,risk_score is an exact additive rule that incl...
6,features,<derived>,"added year, month, month_name, quarter, year_m...",20000,Analytical features; raw columns are retained ...
7,quality,accident_severity/casualties,flagged 'major' accidents with zero casualties...,1725,"Logically inconsistent, but there is no basis ..."
8,outliers,casualties,"IQR rule (1.5×IQR, fences -2.00–6.00) — flagge...",0,Outliers are flagged so analysis can test sens...
9,outliers,vehicles_involved,"IQR rule (1.5×IQR, fences -1.00–7.00) — flagge...",0,Outliers are flagged so analysis can test sens...


## A. Overall accident analysis

In [8]:
pd.Series(A.kpis(df)).to_frame("value")

,value
accidents,20000.000000
casualties,34529.000000
fatal,2987.000000
major,5988.000000
minor,11025.000000
fatal_pct,14.935000
avg_casualties,1.726450
avg_vehicles,2.992000
avg_risk,0.437585
casualties_in_fatal,8966.000000


In [9]:
yt = A.yearly_trend(df); yt

,year,accidents,fatal,casualties,first_date,last_date,days_observed,accidents_per_day,fatal_rate,is_partial,change_in_daily_rate_pct
0,2022,6117,909,10452,2022-01-01,2022-12-31,365,16.758904,14.860226,False,NaN
1,2023,6088,891,10547,2023-01-01,2023-12-31,365,16.679452,14.635348,False,-0.474089
2,2024,6061,925,10442,2024-01-01,2024-12-31,366,16.560109,15.261508,False,-0.715508
3,2025,1734,262,3088,2025-01-01,2025-04-15,105,16.514286,15.109573,True,-0.276711


In [10]:
A.same_period_comparison(df)

,year,accidents,fatal,fatal_rate,window,change_pct
0,2022,1764,281,15.929705,1 Jan – 15 Apr,NaN
1,2023,1761,269,15.275412,1 Jan – 15 Apr,-0.170068
2,2024,1713,255,14.886165,1 Jan – 15 Apr,-2.725724
3,2025,1734,262,15.109573,1 Jan – 15 Apr,1.225919


2025 covers only 1 Jan – 15 Apr, so annual totals are not comparable. On a per-day basis and on the same calendar window, accident frequency is flat across years.

In [11]:
V.lines(A.monthly_series(df), "month_start", "accidents_per_day", y_title="Accidents per day", value_fmt=".1f").show()

## B. Geographic analysis

In [12]:
A.group_summary(df, "state").sort_values("accidents", ascending=False)

,state,accidents,fatal,major,minor,casualties,avg_casualties,avg_vehicles,avg_risk,avg_condition_risk,share_pct,casualty_share_pct,fatal_rate,ci_low,ci_high,reliable
2,Maharashtra,5009,757,1527,2725,8716,1.740068,3.007387,0.435097,0.404871,25.045,25.242550,15.112797,14.147656,16.131408,True
3,Punjab,2577,378,780,1419,4377,1.698487,2.978269,0.441618,0.412282,12.885,12.676301,14.668219,13.354862,16.086755,True
4,Tamil Nadu,2575,371,794,1410,4567,1.773592,3.010874,0.437573,0.408757,12.875,13.226563,14.407767,13.104399,15.817171,True
6,West Bengal,2559,377,737,1445,4508,1.761626,3.020320,0.434584,0.405119,12.795,13.055692,14.732317,13.411970,16.158390,True
1,Karnataka,2438,384,707,1347,4128,1.693191,2.970878,0.440217,0.408716,12.190,11.955168,15.750615,14.358646,17.250346,True
0,Delhi,2433,382,691,1360,4175,1.715988,2.966708,0.437115,0.405754,12.165,12.091286,15.700781,14.309374,17.200327,True
5,Telangana,2409,338,752,1319,4058,1.684516,2.971357,0.439456,0.411395,12.045,11.752440,14.030718,12.701018,15.474950,True


In [13]:
cs = A.group_summary(df, "city").sort_values("accidents", ascending=False); cs

,city,accidents,fatal,major,minor,casualties,avg_casualties,avg_vehicles,avg_risk,avg_condition_risk,share_pct,casualty_share_pct,fatal_rate,ci_low,ci_high,reliable
1,Chandigarh,2577,378,780,1419,4377,1.698487,2.978269,0.441618,0.412282,12.885,12.676301,14.668219,13.354862,16.086755,True
2,Chennai,2575,371,794,1410,4567,1.773592,3.010874,0.437573,0.408757,12.875,13.226563,14.407767,13.104399,15.817171,True
5,Kolkata,2559,377,737,1445,4508,1.761626,3.020320,0.434584,0.405119,12.795,13.055692,14.732317,13.411970,16.158390,True
7,Pune,2517,398,729,1390,4419,1.755662,2.993643,0.435836,0.404211,12.585,12.797938,15.812475,14.439328,17.289818,True
6,Mumbai,2492,359,798,1335,4297,1.724318,3.021268,0.434350,0.405538,12.460,12.444612,14.406100,13.082160,15.839608,True
0,Bangalore,2438,384,707,1347,4128,1.693191,2.970878,0.440217,0.408716,12.190,11.955168,15.750615,14.358646,17.250346,True
3,Delhi,2433,382,691,1360,4175,1.715988,2.966708,0.437115,0.405754,12.165,12.091286,15.700781,14.309374,17.200327,True
4,Hyderabad,2409,338,752,1319,4058,1.684516,2.971357,0.439456,0.411395,12.045,11.752440,14.030718,12.701018,15.474950,True


In [14]:
V.rate_ci(cs, "city", overall=A.kpis(df)["fatal_pct"]).show()
A.fatal_rate_test(df, "city")

{'test': 'Fatality rate differs by city (χ²)',
 'statistic': np.float64(6.814856650546174),
 'dof': 7,
 'p_value': np.float64(0.4484083270699457),
 'effect_size': np.float64(0.01845922079957084),
 'effect_name': "Cramér's V",
 'n': 20000,
 'valid': True,
 'reason': ''}

## C. Temporal analysis

In [15]:
A.per_slot_rate(df)[["time_of_day", "accidents", "hours_in_bucket", "accidents_per_hour", "fatal_rate"]]

,time_of_day,accidents,hours_in_bucket,accidents_per_hour,fatal_rate
2,Morning,5734,7,819.142857,14.788978
0,Afternoon,4180,5,836.000000,14.688995
1,Evening,3362,4,840.500000,14.901844
3,Night,6724,8,840.500000,15.229030


In [16]:
A.weekday_weekend(df)[["day_type", "accidents", "calendar_days", "accidents_per_day", "fatal_rate", "avg_casualties"]]

,day_type,accidents,calendar_days,accidents_per_day,fatal_rate,avg_casualties
0,Weekday,14277,857,16.659277,14.877075,1.725573
1,Weekend,5723,344,16.636628,15.079504,1.728639


In [17]:
A.group_summary(df, "peak_label")[["peak_label", "accidents", "fatal_rate", "avg_casualties", "avg_risk"]]

,peak_label,accidents,fatal_rate,avg_casualties,avg_risk
0,Off-peak,15052,14.928249,1.723160,0.382952
1,Peak hour,4948,14.955538,1.736459,0.603779


In [18]:
m = A.hour_dow_matrix(df); V.heatmap(m, zmin=0, x_title="Hour").show()

In [19]:
A.group_summary(df, "season", config.SEASON_ORDER)

,season,accidents,fatal,major,minor,casualties,avg_casualties,avg_vehicles,avg_risk,avg_condition_risk,share_pct,casualty_share_pct,fatal_rate,ci_low,ci_high,reliable
3,Winter,3909,584,1148,2177,6789,1.736761,3.023024,0.440586,0.410706,19.545,19.661734,14.939882,13.856810,16.091795,True
2,Summer,5399,842,1606,2951,9440,1.748472,3.001297,0.435581,0.404408,26.995,27.339338,15.595481,14.652204,16.587681,True
0,Monsoon,6118,896,1875,3347,10347,1.691239,2.974992,0.435959,0.406669,30.590,29.966115,14.645309,13.781550,15.553438,True
1,Post-monsoon,4574,665,1359,2550,7953,1.738741,2.977263,0.439561,0.410483,22.870,23.032813,14.538697,13.546928,15.589980,True


## D. Severity analysis

In [20]:
for col in ["road_type", "weather", "traffic_density", "time_of_day", "vehicles_involved"]:
    t = A.fatal_rate_test(df, col)
    print(f"{col:18s} p = {t['p_value']:.3f}  Cramér's V = {t['effect_size']:.3f}")

road_type          p = 0.362  Cramér's V = 0.010
weather            p = 0.955  Cramér's V = 0.002
traffic_density    p = 0.966  Cramér's V = 0.002
time_of_day        p = 0.860  Cramér's V = 0.006
vehicles_involved  p = 0.805  Cramér's V = 0.009


In [21]:
V.severity_stack(A.severity_mix(df, "road_type"), "road_type").show()

In [22]:
df.groupby("accident_severity")["casualties"].describe()

,count,mean,std,min,25%,50%,75%,max
accident_severity,,,,,,,,
fatal,2987.0,3.001674,1.416224,1.0,2.0,3.0,4.0,5.0
major,5988.0,1.494823,1.376437,0.0,0.0,1.0,2.0,5.0
minor,11025.0,1.506757,1.390140,0.0,0.0,1.0,2.0,5.0


In [23]:
pd.crosstab(df["accident_severity"], df["casualties"])

casualties,0,1,2,3,4,5
accident_severity,,,,,,
fatal,0,597,604,580,609,597
major,1725,1764,1134,726,454,185
minor,3200,3150,2131,1370,780,394


Fatal accidents average ~3 casualties and *always* have casualties = vehicles involved; 'major' and 'minor' have identical casualty profiles and 1,725 'major' accidents have zero casualties — the severity label is not a reliable injury scale.

## E. Cause analysis

In [24]:
cz = A.group_summary(df, "cause").sort_values("accidents", ascending=False); cz

,cause,accidents,fatal,major,minor,casualties,avg_casualties,avg_vehicles,avg_risk,avg_condition_risk,share_pct,casualty_share_pct,fatal_rate,ci_low,ci_high,reliable
0,distraction,4026,542,1244,2240,6856,1.702931,3.001490,0.429297,0.402372,20.130,19.855773,13.462494,12.442922,14.551725,True
2,overspeeding,4025,603,1203,2219,6943,1.724969,3.007702,0.440186,0.410224,20.125,20.107736,14.981366,13.912227,16.117285,True
4,weather,3997,601,1205,2191,6953,1.739555,2.963723,0.439742,0.409670,19.985,20.136697,15.036277,13.961800,16.177897,True
1,drunk driving,3978,618,1164,2196,6869,1.726747,2.984163,0.438964,0.407893,19.890,19.893423,15.535445,14.443067,16.694322,True
3,poor road,3974,623,1172,2179,6908,1.738299,3.002768,0.439796,0.408468,19.870,20.006371,15.676900,14.579692,16.840400,True


In [25]:
A.chi2_independence(df, "cause", "accident_severity")

{'test': 'χ² independence: cause × accident_severity',
 'statistic': np.float64(10.80904796714973),
 'dof': 8,
 'p_value': np.float64(0.21275531533581055),
 'effect_size': np.float64(0.016438558305969027),
 'effect_name': "Cramér's V",
 'n': 20000,
 'valid': True,
 'reason': ''}

## F. Weather and visibility

In [26]:
A.group_summary(df, "weather")

,weather,accidents,fatal,major,minor,casualties,avg_casualties,avg_vehicles,avg_risk,avg_condition_risk,share_pct,casualty_share_pct,fatal_rate,ci_low,ci_high,reliable
0,clear,6690,992,2017,3681,11491,1.717638,2.993274,0.236876,0.207220,33.450,33.279272,14.828102,13.996710,15.699862,True
1,fog,6633,995,1975,3663,11402,1.718981,2.972411,0.589123,0.559136,33.165,33.021518,15.000754,14.161697,15.880326,True
2,rain,6677,1000,1996,3681,11636,1.742699,3.010184,0.488146,0.458192,33.385,33.699209,14.976786,14.141009,15.852840,True


In [27]:
A.group_summary(df, "visibility", config.LEVEL_ORDER)

,visibility,accidents,fatal,major,minor,casualties,avg_casualties,avg_vehicles,avg_risk,avg_condition_risk,share_pct,casualty_share_pct,fatal_rate,ci_low,ci_high,reliable
1,low,9987,1486,2988,5513,17187,1.720937,2.986783,0.589176,0.559427,49.935,49.775551,14.879343,14.194875,15.590819,True
2,medium,3323,509,983,1831,5851,1.760758,3.005116,0.386067,0.355432,16.615,16.945177,15.317484,14.133042,16.582021,True
0,high,6690,992,2017,3681,11491,1.717638,2.993274,0.236876,0.207220,33.450,33.279272,14.828102,13.996710,15.699862,True


In [28]:
A.temperature_bands(df)

,temp_band,accidents,fatal,major,minor,casualties,avg_casualties,avg_vehicles,avg_risk,avg_condition_risk,share_pct,casualty_share_pct,fatal_rate,ci_low,ci_high,reliable
3,≤19 °C,3738,546,1108,2084,6506,1.740503,2.998930,0.437788,0.408574,18.690,18.842133,14.606742,13.510891,15.775263,True
0,20–24 °C,3853,562,1120,2171,6593,1.711134,2.975344,0.439385,0.410213,19.265,19.094095,14.586037,13.506802,15.735817,True
1,25–29 °C,3858,564,1180,2114,6715,1.740539,3.015552,0.437247,0.408009,19.290,19.447421,14.618974,13.539343,15.768992,True
2,30–34 °C,3956,616,1210,2130,6843,1.729778,2.999747,0.436552,0.405410,19.780,19.818124,15.571284,14.474871,16.734496,True
4,≥35 °C,4595,699,1370,2526,7872,1.713166,2.973885,0.437084,0.406681,22.975,22.798228,15.212187,14.202865,16.279626,True


## G. Road infrastructure

In [29]:
for col in ["road_type", "lanes", "signal_label", "traffic_density"]:
    display(A.group_summary(df, col)[[col, "accidents", "fatal_rate", "ci_low", "ci_high", "avg_casualties"]])

,road_type,accidents,fatal_rate,ci_low,ci_high,avg_casualties
0,highway,6616,14.857920,14.021278,15.735348,1.725967
1,rural,6639,14.535322,13.708010,15.403651,1.721795
2,urban,6745,15.404003,14.562228,16.285162,1.731505


,lanes,accidents,fatal_rate,ci_low,ci_high,avg_casualties
0,1,3382,14.429332,13.285414,15.653964,1.708161
1,2,3270,14.984709,13.802484,16.249107,1.734862
2,3,3334,15.656869,14.462960,16.929827,1.740852
3,4,3383,15.311853,14.137797,16.564599,1.704996
4,5,3363,15.165031,13.992562,16.416991,1.768064
5,6,3268,14.045288,12.896191,15.278812,1.701652


,signal_label,accidents,fatal_rate,ci_low,ci_high,avg_casualties
0,No signal,10003,15.155453,14.466122,15.871537,1.706288
1,Signalised,9997,14.714414,14.033549,15.422387,1.746624


,traffic_density,accidents,fatal_rate,ci_low,ci_high,avg_casualties
0,high,7034,14.856412,14.044449,15.706739,1.732585
1,low,7067,15.013443,14.199650,15.865251,1.720391
2,medium,5899,14.934735,14.047999,15.867110,1.726394


In [30]:
pd.crosstab(df["hour"], df["traffic_density"], normalize="index").round(2).T

hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
traffic_density,,,,,,,,,,,,,,,,,,,,,,,,
high,0.11,0.12,0.09,0.1,0.11,0.38,0.32,0.34,0.62,0.59,0.61,0.36,0.33,0.31,0.34,0.35,0.29,0.6,0.62,0.59,0.32,0.30,0.31,0.34
low,0.71,0.69,0.73,0.7,0.71,0.33,0.34,0.31,0.11,0.10,0.08,0.34,0.31,0.32,0.33,0.35,0.37,0.1,0.08,0.10,0.33,0.32,0.36,0.35
medium,0.18,0.19,0.18,0.2,0.18,0.29,0.34,0.34,0.28,0.31,0.31,0.30,0.35,0.37,0.33,0.30,0.34,0.3,0.30,0.31,0.35,0.38,0.33,0.30


Traffic density is tied to the hour (high density concentrates in the peak hours) — one of the few non-random structures in the data.

## H. Festival and peak-hour analysis

In [31]:
A.group_summary(df, "festival_name")

,festival_name,accidents,fatal,major,minor,casualties,avg_casualties,avg_vehicles,avg_risk,avg_condition_risk,share_pct,casualty_share_pct,fatal_rate,ci_low,ci_high,reliable
0,Diwali,31,3,9,19,60,1.935484,3.193548,0.588710,0.569355,0.155,0.173767,9.677419,3.346464e+00,24.899938,True
1,Eid,34,6,5,23,71,2.088235,3.117647,0.602941,0.570588,0.170,0.205624,17.647059,8.348969e+00,33.513737,True
2,Holi,38,13,12,13,78,2.052632,3.210526,0.521053,0.452632,0.190,0.225897,34.210526,2.121242e+01,50.107896,True
3,New Year,12,0,4,8,25,2.083333,3.666667,0.520833,0.520833,0.060,0.072403,0.000000,2.775558e-15,24.249401,False
4,No Festival,19885,2965,5958,10962,34295,1.724667,2.990646,0.436857,0.407035,99.425,99.322309,14.910737,1.442244e+01,15.412589,True


In [32]:
pd.DataFrame(p["festival"]["calendar_coverage"])

,festival,reference_date,records_on_date,labelled_within_1_day
0,Diwali,2022-10-24,14,0
1,Diwali,2023-11-12,12,12
2,Diwali,2024-10-31,9,19
3,Diwali,2024-11-01,19,19
4,Holi,2022-03-18,9,0
5,Holi,2023-03-08,15,15
6,Holi,2024-03-25,23,23
7,Holi,2025-03-14,16,0
8,Eid,2022-05-03,19,0
9,Eid,2023-04-22,20,20


## I. Risk score

In [33]:
print(RISK_FORMULA_TEXT)
df.groupby("accident_severity")[["risk_score", "condition_risk"]].mean()

risk_score = min(1, 0.10 + 0.20·[severity = fatal] + 0.20·[traffic_density = high] + 0.15·[is_peak_hour] + 0.15·[weather ∈ {fog, rain}] + 0.20·[visibility = low] + 0.10·[festival recorded])


,risk_score,condition_risk
accident_severity,,
fatal,0.607516,0.407549
major,0.406847,0.406847
minor,0.408240,0.408240


In [34]:
A.risk_profiles(df, ["road_type", "weather", "time_of_day", "traffic_density"], min_n=50, top=10)

,road_type,weather,time_of_day,traffic_density,accidents,fatal,avg_casualties,avg_risk,fatal_rate,ci_low,ci_high,vs_overall_pp,ci_excludes_overall,p_value,p_adjusted,significant_after_correction,combinations_tested
98,urban,rain,Afternoon,medium,155,36,1.980645,0.388710,23.225806,17.274592,30.472048,8.290806,True,0.006432,0.681800,False,106
95,urban,fog,Night,medium,167,36,1.718563,0.493114,21.556886,15.997105,28.395784,6.621886,True,0.021981,0.774645,False,106
77,urban,clear,Evening,medium,114,24,1.921053,0.248684,21.052632,14.575300,29.417247,6.117632,False,0.085825,0.842510,False,106
40,rural,clear,Evening,low,54,11,1.944444,0.204630,20.37037,11.774054,32.902307,5.43537,False,0.252903,0.998365,False,106
87,urban,fog,Evening,high,218,44,1.724771,0.820413,20.183486,15.392951,26.006641,5.248486,True,0.036036,0.774645,False,106
19,highway,fog,Morning,low,154,31,1.824675,0.520130,20.12987,14.559403,27.154263,5.19487,False,0.088620,0.842510,False,106
105,urban,rain,Night,high,130,26,1.676923,0.590000,20.0,14.029949,27.692145,5.065,False,0.109523,0.842510,False,106
72,urban,clear,Afternoon,high,151,30,1.913907,0.341060,19.86755,14.286154,26.944056,4.93255,False,0.108588,0.842510,False,106
61,rural,rain,Afternoon,low,135,26,1.762963,0.389259,19.259259,13.495621,26.723964,4.324259,False,0.182579,0.998365,False,106
71,rural,rain,Night,medium,183,35,1.590164,0.371311,19.125683,14.085544,25.435373,4.190683,False,0.119223,0.842510,False,106


## J. Statistical analysis

In [35]:
A.descriptive_stats(df).round(3)

,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
casualties,20000.0,1.726,1.489,0.00,1.00,1.000,3.00,5.00,0.622,-0.593
vehicles_involved,20000.0,2.992,1.416,1.00,2.00,3.000,4.00,5.00,0.011,-1.301
temperature,20000.0,27.580,7.455,15.00,21.00,28.000,34.00,40.00,-0.010,-1.193
lanes,20000.0,3.494,1.705,1.00,2.00,4.000,5.00,6.00,-0.004,-1.264
hour,20000.0,11.487,6.946,0.00,5.00,12.000,18.00,23.00,-0.003,-1.210
risk_score,20000.0,0.438,0.218,0.10,0.25,0.450,0.60,1.00,0.132,-0.597
condition_risk,20000.0,0.408,0.206,0.10,0.25,0.450,0.60,0.90,0.083,-0.771
casualties_per_vehicle,20000.0,0.576,0.409,0.00,0.20,0.667,1.00,1.00,-0.288,-1.528
km_from_city_centre,20000.0,14.460,6.747,0.23,9.28,14.030,19.27,37.24,0.306,-0.486


In [36]:
A.correlation_matrix(df).round(2)

,severity_rank,casualties,vehicles_involved,risk_score,condition_risk,temperature,lanes,hour,is_weekend,is_peak_hour,traffic_signal,festival_flag
severity_rank,1.00,0.23,-0.00,0.22,-0.00,0.01,0.01,0.01,-0.00,0.00,-0.01,0.00
casualties,0.23,1.00,0.51,0.11,0.00,-0.01,0.00,-0.00,0.00,0.00,0.01,0.02
vehicles_involved,-0.00,0.51,1.00,0.00,-0.00,-0.00,0.00,-0.00,0.01,0.00,0.00,0.01
risk_score,0.22,0.11,0.00,1.00,0.94,-0.00,-0.00,0.12,0.00,0.39,0.00,0.04
condition_risk,-0.00,0.00,-0.00,0.94,1.00,-0.00,-0.00,0.12,0.00,0.41,0.00,0.04
temperature,0.01,-0.01,-0.00,-0.00,-0.00,1.00,-0.01,-0.01,-0.01,-0.01,0.00,-0.00
lanes,0.01,0.00,0.00,-0.00,-0.00,-0.01,1.00,0.00,0.01,0.01,-0.00,0.01
hour,0.01,-0.00,-0.00,0.12,0.12,-0.01,0.00,1.00,-0.00,0.17,0.01,0.01
is_weekend,-0.00,0.00,0.01,0.00,0.00,-0.01,0.01,-0.00,1.00,0.00,0.00,0.02
is_peak_hour,0.00,0.00,0.00,0.39,0.41,-0.01,0.01,0.17,0.00,1.00,0.01,-0.01


In [37]:
A.test_battery(df)[["test", "statistic", "p_value", "p_adjusted", "effect_name", "effect_size", "valid", "interpretation"]]

,test,statistic,p_value,p_adjusted,effect_name,effect_size,valid,interpretation
0,Fatality rate differs by city (χ²),6.814857e+00,0.448408,0.991923,Cramér's V,0.018459,True,no evidence of a difference
1,Fatality rate differs by state (χ²),4.865341e+00,0.561197,0.991923,Cramér's V,0.015597,True,no evidence of a difference
2,Fatality rate differs by cause (χ²),9.760888e+00,0.044654,0.323741,Cramér's V,0.022092,True,no evidence of a difference
3,Fatality rate differs by weather (χ²),9.192447e-02,0.955078,0.991923,Cramér's V,0.002144,True,no evidence of a difference
4,Fatality rate differs by visibility (χ²),4.671750e-01,0.791688,0.991923,Cramér's V,0.004833,True,no evidence of a difference
5,Fatality rate differs by road_type (χ²),2.033535e+00,0.361762,0.991923,Cramér's V,0.010083,True,no evidence of a difference
6,Fatality rate differs by traffic_density (χ²),6.842352e-02,0.966367,0.991923,Cramér's V,0.001850,True,no evidence of a difference
7,Fatality rate differs by time_of_day (χ²),7.558299e-01,0.860000,0.991923,Cramér's V,0.006147,True,no evidence of a difference
8,Fatality rate differs by day_type (χ²),1.163187e-01,0.733062,0.991923,Cramér's V,0.002412,True,no evidence of a difference
9,Fatality rate differs by peak_label (χ²),5.632302e-04,0.981066,0.991923,Cramér's V,0.000168,True,no evidence of a difference


In [38]:
A.outlier_summary(df)

,variable,q1,q3,lower_fence,upper_fence,iqr_outliers,z_gt_3,min,max
0,casualties,1.00,3.00,-2.000,6.000,0,0,0.00,5.00
1,vehicles_involved,2.00,4.00,-1.000,7.000,0,0,1.00,5.00
2,temperature,21.00,34.00,1.500,53.500,0,0,15.00,40.00
3,km_from_city_centre,9.28,19.27,-5.705,34.255,28,21,0.23,37.24
4,risk_score,0.25,0.60,-0.275,1.125,0,0,0.10,1.00


## Automatically generated insights

In [39]:
for i in generate_insights(df):
    print(f"[{i['status']:>15}] {i['title']}: {i['value']}\n    {i['detail']}\n")

[    descriptive] Accidents in selection: 20,000
    2,987 fatal (14.9%), 5,988 major and 11,025 minor, involving 34,529 casualties (persons killed or injured — the data does not separate deaths). Average 1.73 casualties per accident.

[not_significant] Accident frequency trend: -1.5% daily rate
    Accidents per day went from 16.76 in 2022 to 16.51 in 2025 (partial year, to 15 Apr). Like-for-like (1 Jan – 15 Apr): 2022: 1,764, 2023: 1,761, 2024: 1,713, 2025: 1,734. Year-to-year differences are not statistically significant (p = 0.801). Raw annual totals are not compared because the last year is incomplete.

[not_significant] Most common city: Chandigarh
    2,577 accidents (12.9%), next: Chennai (2,575). But counts across cities are statistically indistinguishable from an even split (p = 0.077), so 'Chandigarh' leads only marginally. Counts are absolute: without population or vehicle-km exposure data they cannot show which city is more dangerous per traveller.

[not_significant] Highe

## Conclusions

1. The dataset is complete and structurally clean (no missing values outside `festival`, no duplicates, valid coordinates).
2. It is almost certainly **simulated**: near-uniform marginals, every time on the hour, an exact additive formula for `risk_score`, deterministic weather→visibility and casualties ≤ vehicles rules.
3. After multiple-testing correction, **no** location, cause, weather, road, time or infrastructure factor is associated with the fatality rate. Rankings of "deadliest" categories are noise.
4. Several claims in the original notebook (e.g. fog deadliest, highways deadliest, drunk driving highest fatality rate, late-night highest fatality) are **not supported** by the same data.
5. The data is suitable for practising analytics workflow and visual design, not for road-safety policy conclusions.